In [0]:
# Normalization

from pyspark.sql import functions as F
c = spark.read.table("silver_customers")
c.filter("first_name != trim(first_name) OR email != trim(email)").count()

In [0]:
# Duplicate remove (uniqueness)
print("rows:", c.count(), " distinct customer_id:", c.select("customer_id").distinct().count())

In [0]:
# Null keys
c.filter("customer_id IS NULL").count()

In [0]:
print("bronze:", spark.read.table("bronze_customers").count(),
      " -> silver:", c.count())

In [0]:
# e.g. any payment with a method outside the allowed list?
p = spark.read.table("silver_payments")
p.filter("payment_method IS NOT NULL AND lower(payment_method) NOT IN ('card','credit_card','debit_card','paypal','bank transfer','bank_transfer','cash')").count()

In [0]:
spark.read.table("silver_orders").filter("order_date IS NOT NULL AND to_date(order_date) IS NULL").count()

In [0]:
c.select("customer_id","first_name","email","status","_dq_passed","_silver_loaded_at").show(10, truncate=False)

In [0]:
# Non-compliant columns of silver_metadata, with the individual flags
m = spark.read.table("silver_metadata")
m.filter("is_compliant = 0") \
 .select("table_name", "column_name",
         "missing_elements") \
 .show(20, truncate=False)

In [0]:
# quarantine tables
for t in ["customers","orders","products","payments"]:
    q = spark.read.table(f"quarantine_{t}").count()
    print(f"quarantine_{t}: {q} rows")

In [0]:
for tbl, key in [("silver_customers","customer_id"),
                 ("silver_orders","order_id"),
                 ("silver_products","product_id"),
                 ("silver_payments","payment_id"),
                 ("silver_billing_transactions","transaction_id")]:
    df = spark.read.table(tbl)
    total = df.count()
    distinct = df.select(key).distinct().count()
    print(f"{tbl}: rows={total}  distinct {key}={distinct}  ->", "OK" if total == distinct else "DUPLICATES!")

In [0]:
for t in ["customers","orders","products","payments"]:
    s_fail = spark.read.table(f"silver_{t}").filter("_dq_passed = 0").count()
    q_rows = spark.read.table(f"quarantine_{t}").count()
    print(f"{t}: silver _dq_passed=0 = {s_fail}, quarantine = {q_rows} ->",
          "OK" if s_fail == q_rows else "MISMATCH!")

In [0]:
%sql
SHOW TABLES IN workspace. Silver;

In [0]:
%sql
SELECT 'quarantine_customers', COUNT(*) FROM workspace.silver.quarantine_customers
UNION ALL
SELECT 'quarantine_orders', COUNT(*) FROM workspace.silver.quarantine_orders
UNION ALL
SELECT 'quarantine_products', COUNT(*) FROM workspace.silver.quarantine_products
UNION ALL
SELECT 'quarantine_payments', COUNT(*) FROM workspace.silver.quarantine_payments;